In [2]:
# CAPÍTULO VII: ANÁLISIS BIG DATA CON APACHE SPARK
# 7.1. Inicialización del entorno y carga de datos

import os
import sys
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Rutas de los archivos limpios
RUTA_CASOS = r"C:\DatasetsLimpios\Casos_Dengue_Nacional.csv"
RUTA_IPRESS = r"C:\DatasetsLimpios\Establecimientos_Nacional.csv"

# Verificar que los archivos existan
for ruta in [RUTA_CASOS, RUTA_IPRESS]:
    if not os.path.isfile(ruta):
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")
    print(f"Archivo encontrado: {ruta}")

# Crear la sesión de Apache Spark
spark = (
    SparkSession.builder
    .appName("AnalisisBigDataDenguePeru")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("\n=== ENTORNO DE EJECUCIÓN ===")
print("Python:", sys.version.split()[0])
print("Spark:", spark.version)
print("Modo de ejecución: local[*]")

# Cargar los archivos CSV
casos_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(RUTA_CASOS)
)

ipress_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(RUTA_IPRESS)
)

print("\nArchivos cargados correctamente.")

Archivo encontrado: C:\DatasetsLimpios\Casos_Dengue_Nacional.csv
Archivo encontrado: C:\DatasetsLimpios\Establecimientos_Nacional.csv

=== ENTORNO DE EJECUCIÓN ===
Python: 3.14.8
Spark: 4.2.0
Modo de ejecución: local[*]

Archivos cargados correctamente.


In [3]:
# 7.2. Inspección inicial de los datos

print("========== CASOS DE DENGUE ==========")

print("\nColumnas disponibles:")
print(casos_df.columns)

print("\nEsquema de los datos:")
casos_df.printSchema()

print("\nPrimeros 5 registros:")
casos_df.show(5, truncate=False)

print("\n========== ESTABLECIMIENTOS DE SALUD ==========")

print("\nColumnas disponibles:")
print(ipress_df.columns)

print("\nEsquema de los datos:")
ipress_df.printSchema()

print("\nPrimeros 5 registros:")
ipress_df.show(5, truncate=False)

========== CASOS DE DENGUE ==========

Columnas disponibles:
['departamento', 'provincia', 'distrito', 'localidad', 'enfermedad', 'ano', 'semana', 'diagnostic', 'diresa', 'ubigeo', 'localcod', 'edad', 'tipo_edad', 'sexo']

Esquema de los datos:
root
 |-- departamento: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- distrito: string (nullable = true)
 |-- localidad: string (nullable = true)
 |-- enfermedad: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- semana: integer (nullable = true)
 |-- diagnostic: string (nullable = true)
 |-- diresa: integer (nullable = true)
 |-- ubigeo: integer (nullable = true)
 |-- localcod: string (nullable = true)
 |-- edad: double (nullable = true)
 |-- tipo_edad: string (nullable = true)
 |-- sexo: string (nullable = true)


Primeros 5 registros:
+------------+-------------+----------+-----------------+---------------------------+----+------+----------+------+------+--------+----+---------+----+
|departamento|prov

In [4]:
# 7.3. Preparación del conjunto de datos para el análisis

# Mantener los datos en memoria para reducir el costo de
# volver a leer el CSV durante las consultas comparativas.
casos_df.cache()

# Materializar la caché antes de medir las consultas.
total_verificado = casos_df.count()

# Registrar el DataFrame como una vista SQL temporal.
casos_df.createOrReplaceTempView("casos_dengue")

print("Conjunto de datos preparado correctamente.")
print(f"Total de registros verificados: {total_verificado:,}")
print("Vista temporal creada: casos_dengue")

Conjunto de datos preparado correctamente.
Total de registros verificados: 1,029,421
Vista temporal creada: casos_dengue


In [5]:
# 7.4. Agregación anual con DataFrames y Spark SQL

import time
from pyspark.sql import functions as F

REPETICIONES = 3


# A. Consulta mediante DataFrames
def consulta_dataframe():
    return (
        casos_df
        .groupBy("ano")
        .agg(F.count("*").alias("total_casos"))
        .orderBy("ano")
    )


# B. Consulta mediante Spark SQL
def consulta_spark_sql():
    return spark.sql("""
        SELECT
            ano,
            COUNT(*) AS total_casos
        FROM casos_dengue
        GROUP BY ano
        ORDER BY ano
    """)

# Función para medir una consulta que devuelve resultados
def medir_consulta(funcion, repeticiones=REPETICIONES):
    tiempos = []
    resultado = None

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        resultado = funcion().collect()
        fin = time.perf_counter()
        tiempos.append(fin - inicio)

    return tiempos, resultado

# Ejecutar y medir ambos métodos
tiempos_df, resultado_df = medir_consulta(consulta_dataframe)
tiempos_sql, resultado_sql = medir_consulta(consulta_spark_sql)

# Mostrar resultados anuales
print("========== CASOS POR AÑO ==========")
print("Resultados de DataFrames:")
for fila in resultado_df:
    print(f"Año: {fila['ano']} | Casos: {fila['total_casos']:,}")

# Verificar que ambas consultas produzcan los mismos resultados
equivalentes = resultado_df == resultado_sql

print("\n========== VALIDACIÓN ==========")
print("¿Ambas consultas producen los mismos resultados?:", equivalentes)
print("Años agrupados:", len(resultado_df))

# Mostrar tiempos individuales y promedio
print("\n========== TIEMPOS DE EJECUCIÓN ==========")

print("\nDataFrames:")
for i, tiempo in enumerate(tiempos_df, start=1):
    print(f"Ejecución {i}: {tiempo:.4f} segundos")
print(f"Promedio DataFrames: {sum(tiempos_df) / len(tiempos_df):.4f} segundos")

print("\nSpark SQL:")
for i, tiempo in enumerate(tiempos_sql, start=1):
    print(f"Ejecución {i}: {tiempo:.4f} segundos")
print(f"Promedio Spark SQL: {sum(tiempos_sql) / len(tiempos_sql):.4f} segundos")

========== CASOS POR AÑO ==========
Resultados de DataFrames:
Año: 2000 | Casos: 5,557
Año: 2001 | Casos: 23,526
Año: 2002 | Casos: 8,086
Año: 2003 | Casos: 3,349
Año: 2004 | Casos: 9,547
Año: 2005 | Casos: 5,640
Año: 2006 | Casos: 4,022
Año: 2007 | Casos: 6,344
Año: 2008 | Casos: 12,824
Año: 2009 | Casos: 13,407
Año: 2010 | Casos: 16,842
Año: 2011 | Casos: 28,084
Año: 2012 | Casos: 28,505
Año: 2013 | Casos: 13,091
Año: 2014 | Casos: 17,234
Año: 2015 | Casos: 35,816
Año: 2016 | Casos: 25,160
Año: 2017 | Casos: 68,290
Año: 2018 | Casos: 4,698
Año: 2019 | Casos: 15,287
Año: 2020 | Casos: 47,932
Año: 2021 | Casos: 44,791
Año: 2022 | Casos: 63,217
Año: 2023 | Casos: 256,641
Año: 2024 | Casos: 271,531

========== VALIDACIÓN ==========
¿Ambas consultas producen los mismos resultados?: True
Años agrupados: 25

========== TIEMPOS DE EJECUCIÓN ==========

DataFrames:
Ejecución 1: 0.9333 segundos
Ejecución 2: 0.3464 segundos
Ejecución 3: 0.2737 segundos
Promedio DataFrames: 0.5178 segundos

Spar

In [6]:
# 7.5. Recuento de registros de los conjuntos de datos

inicio = time.perf_counter()
total_casos = casos_df.count()
tiempo_conteo_casos = time.perf_counter() - inicio

inicio = time.perf_counter()
total_ipress = ipress_df.count()
tiempo_conteo_ipress = time.perf_counter() - inicio

print("========== RESULTADOS DEL CONTEO ==========")
print(f"Registros de dengue: {total_casos:,}")
print(f"Tiempo de conteo de dengue: {tiempo_conteo_casos:.3f} segundos")

print(f"\nEstablecimientos de salud: {total_ipress:,}")
print(f"Tiempo de conteo de establecimientos: {tiempo_conteo_ipress:.3f} segundos")

========== RESULTADOS DEL CONTEO ==========
Registros de dengue: 1,029,421
Tiempo de conteo de dengue: 0.160 segundos

Establecimientos de salud: 20,793
Tiempo de conteo de establecimientos: 0.126 segundos


# 7.6.  Ejecutar esta consulta en SSMS22 3 veces y capturar su valor "elapsed time"

USE DengueCapacidadSanitaria;
GO

SET NOCOUNT ON;
SET STATISTICS TIME ON;

SELECT
    ano,
    COUNT_BIG(*) AS total_casos
FROM dbo.Caso_Dengue
GROUP BY ano
ORDER BY ano;

SET STATISTICS TIME OFF;
GO

In [7]:
# 7.7. Análisis de casos por departamento

from pyspark.sql import functions as F

casos_por_departamento = (
    casos_df
    .groupBy("departamento")
    .agg(
        F.count("*").alias("total_casos"),
        F.countDistinct("ubigeo").alias("ubigeos_registrados")
    )
    .orderBy(F.desc("total_casos"))
)

print("========== TOP 15 DEPARTAMENTOS ==========")
casos_por_departamento.show(15, truncate=False)

print("\n========== RESUMEN DE LA DISTRIBUCIÓN ==========")
print("Departamentos o valores geográficos distintos:")
print(
    casos_df
    .select("departamento")
    .distinct()
    .count()
)

print("Total de registros:")
print(casos_df.count())

========== TOP 15 DEPARTAMENTOS ==========
+-------------+-----------+-------------------+
|departamento |total_casos|ubigeos_registrados|
+-------------+-----------+-------------------+
|PIURA        |237961     |62                 |
|LIMA         |122854     |81                 |
|LORETO       |107074     |53                 |
|LA LIBERTAD  |100810     |44                 |
|ICA          |68620      |39                 |
|LAMBAYEQUE   |56383      |37                 |
|UCAYALI      |52621      |19                 |
|TUMBES       |45471      |13                 |
|SAN MARTIN   |44518      |78                 |
|ANCASH       |37661      |17                 |
|MADRE DE DIOS|32920      |11                 |
|CAJAMARCA    |29711      |53                 |
|JUNIN        |26110      |19                 |
|AMAZONAS     |16805      |25                 |
|CUSCO        |15210      |21                 |
+-------------+-----------+-------------------+
only showing top 15 rows

========== RESUMEN 

In [8]:
# 7.8. Análisis de casos según diagnóstico

diagnosticos = (
    casos_df
    .groupBy("diagnostic", "enfermedad")
    .agg(F.count("*").alias("total_registros"))
    .orderBy(F.desc("total_registros"))
)

print("========== DISTRIBUCIÓN POR DIAGNÓSTICO ==========")
diagnosticos.show(20, truncate=False)

print("\n========== CANTIDAD DE CATEGORÍAS ==========")
print("Códigos de diagnóstico distintos:")
print(casos_df.select("diagnostic").distinct().count())

print("Descripciones de enfermedad distintas:")
print(casos_df.select("enfermedad").distinct().count())

========== DISTRIBUCIÓN POR DIAGNÓSTICO ==========
+----------+---------------------------+---------------+
|diagnostic|enfermedad                 |total_registros|
+----------+---------------------------+---------------+
|A97.0     |DENGUE SIN SIGNOS DE ALARMA|915243         |
|A97.1     |DENGUE CON SIGNOS DE ALARMA|110166         |
|A97.2     |DENGUE GRAVE               |4012           |
+----------+---------------------------+---------------+


========== CANTIDAD DE CATEGORÍAS ==========
Códigos de diagnóstico distintos:
3
Descripciones de enfermedad distintas:
3


In [9]:
# 7.9. Estadística descriptiva de la edad

print("========== ESTADÍSTICAS DE EDAD ==========")

casos_df.select(
    F.count("edad").alias("edades_no_nulas"),
    F.mean("edad").alias("edad_promedio"),
    F.stddev("edad").alias("desviacion_estandar"),
    F.min("edad").alias("edad_minima"),
    F.max("edad").alias("edad_maxima")
).show(truncate=False)

print("========== REGISTROS POR SEXO ==========")

casos_df.groupBy("sexo").agg(
    F.count("*").alias("total_registros")
).orderBy(
    F.desc("total_registros")
).show(truncate=False)

print("========== REGISTROS POR UNIDAD DE EDAD ==========")

casos_df.groupBy("tipo_edad").agg(
    F.count("*").alias("total_registros")
).orderBy(
    F.desc("total_registros")
).show(truncate=False)

========== ESTADÍSTICAS DE EDAD ==========
+---------------+------------------+-------------------+-----------+-----------+
|edades_no_nulas|edad_promedio     |desviacion_estandar|edad_minima|edad_maxima|
+---------------+------------------+-------------------+-----------+-----------+
|1029407        |30.596995428435985|18.996258511396444 |0.0        |110.0      |
+---------------+------------------+-------------------+-----------+-----------+

========== REGISTROS POR SEXO ==========
+----+---------------+
|sexo|total_registros|
+----+---------------+
|F   |553677         |
|M   |475744         |
+----+---------------+

========== REGISTROS POR UNIDAD DE EDAD ==========
+---------+---------------+
|tipo_edad|total_registros|
+---------+---------------+
|A        |1029421        |
+---------+---------------+



# 7.10. Interpretación de resultados
Distribución geográfica

Piura concentra la mayor cantidad de registros, con 237,961, seguida de Lima con 122,854, Loreto con 107,074 y La Libertad con 100,810. En conjunto, estos cuatro departamentos reúnen una proporción importante de los registros históricos analizados.

Estos resultados describen el volumen de registros por departamento entre 2000 y 2024; no deben interpretarse por sí solos como tasas de incidencia poblacional, ya que no se han dividido entre las poblaciones correspondientes.

La categoría predominante es el dengue sin signos de alarma, que representa aproximadamente el 88.9 % de los registros. El dengue con signos de alarma representa alrededor del 10.7 %, mientras que el dengue grave corresponde aproximadamente al 0.4 %.

Estadística descriptiva de edad y sexo: 

- Edad promedio registrada: 30.60 años.

- Desviación estándar: 19.00 años.

- Edad mínima y máxima: 0 y 110 años.

- Registros sin edad válida: 14.

- Sexo femenino: 553,677 registros (aprox. 53.8 %).

- Sexo masculino: 475,744 registros (aprox. 46.2 %).

La media de edad debe interpretarse como una estadística descriptiva de los registros disponibles, no como una medida del riesgo individual de contraer dengue. Asimismo, el predominio de registros femeninos no permite concluir que exista mayor riesgo en mujeres sin considerar población, cobertura de notificación y otros factores.

In [10]:
# 7.11. Resumen cuantitativo de los resultados

from pyspark.sql import functions as F

total = casos_df.count()

print("========== INDICADORES GENERALES ==========")
print(f"Total de registros: {total:,}")

print("\n========== PARTICIPACIÓN POR DIAGNÓSTICO ==========")

(
    casos_df
    .groupBy("diagnostic", "enfermedad")
    .agg(F.count("*").alias("registros"))
    .withColumn(
        "porcentaje",
        F.round(F.col("registros") * 100 / total, 2)
    )
    .orderBy(F.desc("registros"))
    .show(truncate=False)
)

print("\n========== AÑOS CON MÁS REGISTROS ==========")

(
    casos_df
    .groupBy("ano")
    .agg(F.count("*").alias("registros"))
    .orderBy(F.desc("registros"))
    .show(5, truncate=False)
)

print("\n========== PARTICIPACIÓN POR SEXO ==========")

(
    casos_df
    .groupBy("sexo")
    .agg(F.count("*").alias("registros"))
    .withColumn(
        "porcentaje",
        F.round(F.col("registros") * 100 / total, 2)
    )
    .orderBy(F.desc("registros"))
    .show(truncate=False)
)

========== INDICADORES GENERALES ==========
Total de registros: 1,029,421

========== PARTICIPACIÓN POR DIAGNÓSTICO ==========
+----------+---------------------------+---------+----------+
|diagnostic|enfermedad                 |registros|porcentaje|
+----------+---------------------------+---------+----------+
|A97.0     |DENGUE SIN SIGNOS DE ALARMA|915243   |88.91     |
|A97.1     |DENGUE CON SIGNOS DE ALARMA|110166   |10.7      |
|A97.2     |DENGUE GRAVE               |4012     |0.39      |
+----------+---------------------------+---------+----------+


========== AÑOS CON MÁS REGISTROS ==========
+----+---------+
|ano |registros|
+----+---------+
|2024|271531   |
|2023|256641   |
|2017|68290    |
|2022|63217    |
|2020|47932    |
+----+---------+
only showing top 5 rows

========== PARTICIPACIÓN POR SEXO ==========
+----+---------+----------+
|sexo|registros|porcentaje|
+----+---------+----------+
|F   |553677   |53.79     |
|M   |475744   |46.21     |
+----+---------+----------+



In [11]:
# 7.12. Inspección del plan de ejecución y particiones

import os

print("========== TAMAÑO DE LOS ARCHIVOS ==========")
print(
    "CSV de dengue:",
    f"{os.path.getsize(RUTA_CASOS) / (1024**2):.2f} MB"
)
print(
    "CSV de establecimientos:",
    f"{os.path.getsize(RUTA_IPRESS) / (1024**2):.2f} MB"
)

print("\n========== PARTICIONES DEL DATAFRAME ==========")
print("Particiones de casos_df:", casos_df.rdd.getNumPartitions())
print("Núcleos lógicos disponibles para Spark:", spark.sparkContext.defaultParallelism)

print("\n========== PLAN DE EJECUCIÓN DE LA AGREGACIÓN ==========")

(
    casos_df
    .groupBy("ano")
    .agg(F.count("*").alias("total_casos"))
    .explain("formatted")
)

========== TAMAÑO DE LOS ARCHIVOS ==========
CSV de dengue: 104.46 MB
CSV de establecimientos: 8.18 MB

========== PARTICIONES DEL DATAFRAME ==========
Particiones de casos_df: 12
Núcleos lógicos disponibles para Spark: 12

========== PLAN DE EJECUCIÓN DE LA AGREGACIÓN ==========
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- InMemoryTableScan (1)
               +- InMemoryRelation (2)
                     +- Scan csv  (3)


(1) InMemoryTableScan
Output [1]: [ano#22]
Arguments: [ano#22]

(2) InMemoryRelation
Arguments: [departamento#17, provincia#18, distrito#19, localidad#20, enfermedad#21, ano#22, semana#23, diagnostic#24, diresa#25, ubigeo#26, localcod#27, edad#28, tipo_edad#29, sexo#30], StorageLevel(disk, memory, deserialized, 1 replicas)

(3) Scan csv 
Output [14]: [departamento#17, provincia#18, distrito#19, localidad#20, enfermedad#21, ano#22, semana#23, diagnostic#24, diresa#25, ubigeo#26, localcod#27, e